# M1.0b Primer: Line, Surface and Volume Integrals

Book pointer: chapter 1, section 1.3 (integration).

Second of four primer notebooks. Integrals in electromagnetism come in three kinds: along a curve, over a surface, and through a volume. Each is a sum of many small contributions, and each will be computed here first as that literal sum, with the formula written out, before the shortcut appears.

## 1. Warm-up

1. What does $\int_a^b f(x)\,dx$ mean as a sum? What is the error of a midpoint-rule sum with $N$ pieces, roughly?
2. A wire has mass per length $\lambda(\mathbf{r})$. Write its total mass as an integral along the wire.
3. A force $\mathbf{F}$ acts on a particle moving along a path. Write the work done as an integral. Which part of $\mathbf{F}$ contributes?
4. What is flux? If the velocity of water is $\mathbf{v}$, what does $\int_S \mathbf{v}\cdot d\mathbf{S}$ measure?
5. Write the volume element in spherical coordinates and explain where each factor comes from.

## 2. The idea in one paragraph

Every integral is a limit of a sum: cut the domain into small pieces, multiply the function on each piece by the size of the piece, add up, and refine. For a curve the pieces are small displacement vectors $\Delta\mathbf{l}$; for a surface they are small patches with an area and a normal direction, $\Delta\mathbf{S} = \hat{\mathbf{n}}\,\Delta S$; for a volume they are small cells $\Delta v$. When the integrand is a vector field we usually dot it with $\Delta\mathbf{l}$ or $\Delta\mathbf{S}$ first, which picks out the part of the field along the curve or through the surface. The code in this notebook is always the sum, so that the formula and the computation are the same thing.

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (6, 5)
np.set_printoptions(precision=5, suppress=True)

### 3.1 One dimension first: integrals as sums

The midpoint rule cuts $[a, b]$ into $N$ pieces of width $\Delta x = (b - a)/N$, evaluates $f$ at the centre of each piece, and adds $f(x_k)\,\Delta x$. The trapezoid rule evaluates at the ends of each piece and averages. Both converge like $1/N^2$: ten times more pieces, a hundred times smaller error. That rate is worth knowing because it tells you how fine a grid you need.

**Worked example 1.** $\int_0^1 x^2\,dx = 1/3$. With $N = 2$ midpoints at $x = 0.25$ and $0.75$: $(0.0625 + 0.5625)\cdot0.5 = 0.3125$. Error $0.0208$. With $N = 4$ the midpoints are $0.125, 0.375, 0.625, 0.875$ and the sum is $(0.015625 + 0.140625 + 0.390625 + 0.765625)\cdot0.25 = 0.328125$. Error $0.0052$, a quarter of before: doubling $N$ quarters the error.

In [ ]:
def midpoint(f, a, b, N):
    xk = a + (np.arange(N) + 0.5) * (b - a) / N
    return np.sum(f(xk)) * (b - a) / N

for N in (2, 4, 10, 100, 1000):
    val = midpoint(lambda x: x**2, 0, 1, N)
    print(f"N = {N:5d}:  midpoint sum = {val:.7f}   error = {abs(val - 1 / 3):.2e}")

### 3.2 Line integrals (scalar integrand)

A curve $C$ is described by a parametrisation $\mathbf{r}(t)$ for $t$ from $t_a$ to $t_b$. A small change $dt$ moves the point by $d\mathbf{l} = \mathbf{r}'(t)\,dt$, whose length is $dl = |\mathbf{r}'(t)|\,dt$. The arc length of the curve is $\int_C dl$, and the mass of a wire with line density $\lambda$ is $\int_C \lambda\,dl$. In the computer: sample $t$, build the points, take differences between consecutive points as $\Delta\mathbf{l}$, and sum $\lambda|\Delta\mathbf{l}|$.

**Worked example 2.** One turn of a helix $\mathbf{r}(t) = (\cos t, \sin t, t/2\pi)$, $t \in [0, 2\pi]$: it goes once around a unit circle while rising by one unit. $\mathbf{r}'(t) = (-\sin t, \cos t, 1/2\pi)$, so $|\mathbf{r}'| = \sqrt{1 + 1/4\pi^2} = 1.01258$, constant. Arc length $= 2\pi\cdot1.01258 = 6.3622$. (Flatten the helix out: it is the hypotenuse of a right triangle with legs $2\pi$ and $1$, and $\sqrt{4\pi^2 + 1} = 6.3622$.)

**Worked example 3.** A wire along the parabola $y = x^2$ from $x = 0$ to $1$ with density $\lambda = x$. Parametrise by $x$ itself: $\mathbf{r}(x) = (x, x^2)$, $dl = \sqrt{1 + 4x^2}\,dx$, mass $= \int_0^1 x\sqrt{1 + 4x^2}\,dx = \big[(1 + 4x^2)^{3/2}/12\big]_0^1 = (5^{3/2} - 1)/12 = 0.8484$.

In [ ]:
def polyline(r_of_t, t_a, t_b, N=2000):
    t = np.linspace(t_a, t_b, N + 1)
    return r_of_t(t)                                   # (N+1, dim) array of points along the curve

def scalar_line_integral(density, path):
    dl = np.diff(path, axis=0)
    mid = 0.5 * (path[1:] + path[:-1])
    return np.sum(density(mid) * np.linalg.norm(dl, axis=1))

helix = polyline(lambda t: np.stack([np.cos(t), np.sin(t), t / (2 * np.pi)], axis=-1), 0, 2 * np.pi)
print(f"arc length of one helix turn: {scalar_line_integral(lambda p: 1.0, helix):.5f}   exact sqrt(4 pi^2 + 1) = {np.sqrt(4 * np.pi**2 + 1):.5f}")

parabola = polyline(lambda s: np.stack([s, s**2], axis=-1), 0, 1)
print(f"mass of the parabolic wire: {scalar_line_integral(lambda p: p[:, 0], parabola):.5f}   exact (5^1.5 - 1)/12 = {(5**1.5 - 1) / 12:.5f}")

### 3.3 Line integrals of a vector field: work

When the integrand is a vector field we dot it with the step: $\int_C \mathbf{F}\cdot d\mathbf{l} = \int_{t_a}^{t_b} \mathbf{F}(\mathbf{r}(t))\cdot\mathbf{r}'(t)\,dt$. Only the component of $\mathbf{F}$ along the direction of travel counts. If $\mathbf{F}$ is a force this is the work done on a particle that moves along $C$. In chapter 3 it is the negative of the change in electric potential.

This integral depends on the orientation of the curve (reverse it and the sign flips) and, in general, on which curve you take between two endpoints. The exception is the most important case in electrostatics: for some fields the integral depends only on the endpoints. Those fields are called conservative, and the test is that the integral around any closed loop is zero.

**Worked example 4.** $\mathbf{F} = (y, x)$ from $(0, 0)$ to $(1, 1)$. Straight line $\mathbf{r}(t) = (t, t)$: $\mathbf{r}' = (1, 1)$, $\mathbf{F}\cdot\mathbf{r}' = t + t = 2t$, integral $\int_0^1 2t\,dt = 1$. Parabola $\mathbf{r}(t) = (t, t^2)$: $\mathbf{r}' = (1, 2t)$, $\mathbf{F} = (t^2, t)$, $\mathbf{F}\cdot\mathbf{r}' = t^2 + 2t^2 = 3t^2$, integral $1$. Same answer. Indeed $(y, x) = \nabla(xy)$ and the integral is $xy$ at the end minus at the start.

**Worked example 5.** $\mathbf{F} = (-y, x)$ along the same two paths. Straight line: $\mathbf{F}\cdot\mathbf{r}' = -t + t = 0$, integral $0$. Parabola: $\mathbf{F} = (-t^2, t)$, $\mathbf{F}\cdot\mathbf{r}' = -t^2 + 2t^2 = t^2$, integral $1/3$. Different answers for the same endpoints: not conservative. Around the unit circle $\mathbf{r}(t) = (\cos t, \sin t)$: $\mathbf{F}\cdot\mathbf{r}' = \sin^2t + \cos^2t = 1$, integral $2\pi$, not zero.

In [ ]:
def vector_line_integral(F, path):
    """Sum over steps of F(midpoint) . dl."""
    dl = np.diff(path, axis=0)
    mid = 0.5 * (path[1:] + path[:-1])
    return np.sum(F(mid) * dl)

F_grad = lambda p: np.stack([p[:, 1], p[:, 0]], axis=-1)        # (y, x) = grad(xy)
F_rot = lambda p: np.stack([-p[:, 1], p[:, 0]], axis=-1)        # (-y, x)

straight = polyline(lambda t: np.stack([t, t], axis=-1), 0, 1)
parab = polyline(lambda t: np.stack([t, t**2], axis=-1), 0, 1)
circle = polyline(lambda t: np.stack([np.cos(t), np.sin(t)], axis=-1), 0, 2 * np.pi)
print("F = (y, x):   straight", round(vector_line_integral(F_grad, straight), 5), "  parabola", round(vector_line_integral(F_grad, parab), 5), "  unit circle", round(vector_line_integral(F_grad, circle), 5))
print("F = (-y, x):  straight", round(vector_line_integral(F_rot, straight), 5), "  parabola", round(vector_line_integral(F_rot, parab), 5), "  unit circle", round(vector_line_integral(F_rot, circle), 5), " (= 2 pi)")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
xg = np.linspace(-0.2, 1.2, 15); Xg, Yg = np.meshgrid(xg, xg, indexing="ij")
for ax, (Fx, Fy), name in zip(axes, ((Yg, Xg), (-Yg, Xg)), ("F = (y, x): path independent", "F = (-y, x): path dependent")):
    ax.quiver(Xg, Yg, Fx, Fy, color="grey", alpha=0.6, pivot="mid")
    ax.plot(straight[:, 0], straight[:, 1], "C0", lw=2, label="straight"); ax.plot(parab[:, 0], parab[:, 1], "C3", lw=2, label="parabola")
    ax.set_aspect("equal"); ax.set_title(name); ax.legend()
plt.show()

### 3.4 Surface integrals: flux

A surface is parametrised by two variables, $\mathbf{r}(u, v)$. A small rectangle $du\,dv$ in parameter space maps to a small parallelogram on the surface with sides $\partial\mathbf{r}/\partial u\,du$ and $\partial\mathbf{r}/\partial v\,dv$. From notebook 00a, the cross product of two sides gives a vector perpendicular to the parallelogram whose length is its area. So

$$d\mathbf{S} = \left(\frac{\partial\mathbf{r}}{\partial u}\times\frac{\partial\mathbf{r}}{\partial v}\right)du\,dv = \hat{\mathbf{n}}\,dS.$$

That one formula gives both the area element and the normal direction. The flux of $\mathbf{F}$ through the surface is $\int_S \mathbf{F}\cdot d\mathbf{S}$, the amount of $\mathbf{F}$ that pokes through. For a closed surface the convention is that $\hat{\mathbf{n}}$ points outward; if the cross product comes out inward, swap $u$ and $v$ or flip the sign.

**Worked example 6.** A sphere of radius $R$, parametrised by the polar angle $\theta$ and azimuth $\phi$: $\mathbf{r} = R(\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta)$. Then
$\partial\mathbf{r}/\partial\theta = R(\cos\theta\cos\phi, \cos\theta\sin\phi, -\sin\theta)$ and $\partial\mathbf{r}/\partial\phi = R(-\sin\theta\sin\phi, \sin\theta\cos\phi, 0)$. Their cross product works out to $R^2\sin\theta\,(\sin\theta\cos\phi, \sin\theta\sin\phi, \cos\theta) = R^2\sin\theta\,\hat{\mathbf{r}}$. So $dS = R^2\sin\theta\,d\theta\,d\phi$, outward normal $\hat{\mathbf{r}}$. The $\sin\theta$ is there because circles of latitude shrink towards the poles. Integrating $dS$ over the sphere: $R^2\int_0^\pi\sin\theta\,d\theta\int_0^{2\pi}d\phi = R^2\cdot2\cdot2\pi = 4\pi R^2$.

The code checks the cross-product formula against the hand result by finite differences, then integrates the area.

In [ ]:
def sphere(theta, phi, R=1.0):
    return R * np.stack([np.sin(theta) * np.cos(phi), np.sin(theta) * np.sin(phi), np.cos(theta)], axis=-1)

# dS by cross product of finite-difference tangent vectors, at one point.
th, ph, eps = 0.7, 1.1, 1e-6
r_theta = (sphere(th + eps, ph) - sphere(th - eps, ph)) / (2 * eps)
r_phi = (sphere(th, ph + eps) - sphere(th, ph - eps)) / (2 * eps)
dS_vec = np.cross(r_theta, r_phi)
print("dr/dtheta x dr/dphi =", dS_vec)
print("R^2 sin(theta) r_hat =", np.sin(th) * sphere(th, ph))

def surface_patches(r_of_uv, u_range, v_range, nu, nv):
    """Patch centres and vector area elements dS (both (nu*nv, 3)) from a parametrisation."""
    u = u_range[0] + (np.arange(nu) + 0.5) * (u_range[1] - u_range[0]) / nu
    v = v_range[0] + (np.arange(nv) + 0.5) * (v_range[1] - v_range[0]) / nv
    du, dv = (u_range[1] - u_range[0]) / nu, (v_range[1] - v_range[0]) / nv
    U, V = np.meshgrid(u, v, indexing="ij")
    e = 1e-6
    ru = (r_of_uv(U + e, V) - r_of_uv(U - e, V)) / (2 * e)
    rv = (r_of_uv(U, V + e) - r_of_uv(U, V - e)) / (2 * e)
    dS = np.cross(ru, rv) * du * dv
    return r_of_uv(U, V).reshape(-1, 3), dS.reshape(-1, 3)

centres, dS = surface_patches(sphere, (0, np.pi), (0, 2 * np.pi), 200, 400)
print(f"sphere area from patches: {np.linalg.norm(dS, axis=1).sum():.5f}   exact 4 pi = {4 * np.pi:.5f}")

**Worked example 7.** Flux of the radial field $\mathbf{F} = (x, y, z) = \mathbf{r}$ through a sphere of radius $R$. On the sphere $\mathbf{F} = R\hat{\mathbf{r}}$ and $d\mathbf{S} = \hat{\mathbf{r}}R^2\sin\theta\,d\theta\,d\phi$, so $\mathbf{F}\cdot d\mathbf{S} = R^3\sin\theta\,d\theta\,d\phi$ and the flux is $4\pi R^3$. Note that $4\pi R^3 = 3\times\frac43\pi R^3 = 3\times$ volume. Keep that in mind for notebook 00d: the divergence of $\mathbf{r}$ is 3.

**Worked example 8.** Flux of the same field through the upper hemisphere only. By symmetry it is half, $2\pi R^3$. Through the flat disk that closes the hemisphere (at $z = 0$, normal $-\hat{\mathbf{z}}$ to point out of the half-ball), $\mathbf{F}\cdot\hat{\mathbf{n}} = -z = 0$: no flux. So the closed half-ball surface has total flux $2\pi R^3 = 3\times$ its volume. Same pattern.

**Worked example 9.** Flux of the rotation field $(-y, x, 0)$ through the sphere. On the sphere, $\mathbf{F}\cdot\hat{\mathbf{r}} = (-y x + x y + 0)/R = 0$ at every point: the field runs along the surface and never pokes through. Flux zero.

In [ ]:
def flux(F, centres, dS):
    return np.sum(F(centres) * dS)

F_radial = lambda p: p
F_rotation = lambda p: np.stack([-p[:, 1], p[:, 0], np.zeros(len(p))], axis=-1)
print(f"radial field through the unit sphere:    {flux(F_radial, centres, dS):.5f}   exact 4 pi = {4 * np.pi:.5f}")
print(f"rotation field through the unit sphere:  {flux(F_rotation, centres, dS):.2e}   exact 0")

hemi_c, hemi_dS = surface_patches(sphere, (0, np.pi / 2), (0, 2 * np.pi), 100, 400)
disk = lambda s, phi: np.stack([s * np.cos(phi), s * np.sin(phi), np.zeros_like(s)], axis=-1)
disk_c, disk_dS = surface_patches(disk, (0, 1), (0, 2 * np.pi), 100, 400)
disk_dS = -disk_dS * np.sign(disk_dS[:, 2:3])            # force the disk normal to point down, out of the half ball
print(f"radial field through the upper hemisphere: {flux(F_radial, hemi_c, hemi_dS):.5f}   through the closing disk: {flux(F_radial, disk_c, disk_dS):.2e}   exact 2 pi = {2 * np.pi:.5f} and 0")

### 3.5 Volume integrals

A volume integral adds a density times small volumes. In Cartesian coordinates the cell is $dv = dx\,dy\,dz$ and the sum runs over a 3D grid. If the density is a charge density $\rho$, the integral is the total charge. The same sum with $\rho = 1$ gives the volume.

In spherical coordinates the cell between $r$ and $r + dr$, $\theta$ and $\theta + d\theta$, $\phi$ and $\phi + d\phi$ has sides $dr$, $r\,d\theta$ and $r\sin\theta\,d\phi$, so $dv = r^2\sin\theta\,dr\,d\theta\,d\phi$. The factor $r^2\sin\theta$ is the surface element from section 3.4 times the radial thickness $dr$.

**Worked example 10.** A ball of radius $R$ with charge density $\rho = \rho_0(1 - r/R)$, densest at the centre and zero at the surface. Total charge in spherical coordinates, using the angular integrals $\int\sin\theta\,d\theta\,d\phi = 4\pi$:
$$Q = 4\pi\rho_0\int_0^R \left(1 - \frac rR\right)r^2\,dr = 4\pi\rho_0\left(\frac{R^3}{3} - \frac{R^3}{4}\right) = \frac{\pi\rho_0R^3}{3}.$$
With $\rho_0 = R = 1$: $Q = 1.0472$. The code does the same integral two ways: as a Cartesian grid sum over a cube, keeping only cells inside the ball, and as a spherical grid sum where the $r^2\sin\theta$ factor appears explicitly.

In [ ]:
rho0, R = 1.0, 1.0
rho = lambda r: rho0 * (1 - r / R)

# Cartesian: cells of a cube, keep those inside the ball.
for n in (20, 40, 80, 160):
    s = -R + (np.arange(n) + 0.5) * (2 * R / n)
    Xc, Yc, Zc = np.meshgrid(s, s, s, indexing="ij")
    rc = np.sqrt(Xc**2 + Yc**2 + Zc**2)
    Q_cart = np.sum(np.where(rc <= R, rho(rc), 0.0)) * (2 * R / n) ** 3
    print(f"Cartesian grid, {n}^3 cells:  Q = {Q_cart:.5f}   error {abs(Q_cart - np.pi / 3):.1e}")

# Spherical: cells in (r, theta, phi) with volume r^2 sin(theta) dr dtheta dphi.
nr, nt, np_ = 100, 50, 100
r = (np.arange(nr) + 0.5) * R / nr; th = (np.arange(nt) + 0.5) * np.pi / nt; ph = (np.arange(np_) + 0.5) * 2 * np.pi / np_
Rg, Tg, Pg = np.meshgrid(r, th, ph, indexing="ij")
dv = Rg**2 * np.sin(Tg) * (R / nr) * (np.pi / nt) * (2 * np.pi / np_)
Q_sph = np.sum(rho(Rg) * dv)
print(f"spherical grid:  Q = {Q_sph:.5f}   exact pi/3 = {np.pi / 3:.5f}")

The Cartesian sum converges slowly because the ball's surface is staircased: the error is set by how well the cubes fit the sphere, not by the smoothness of $\rho$. The spherical grid fits the ball exactly and is accurate at once. That is the general lesson of section 1.5: match the coordinates to the geometry.

### Exercise: flux through a cube

Replace the body of `flux_through_cube` so that it computes the outward flux of a field through the six faces of the cube $|x|, |y|, |z| \le a$ by sampling each face on a grid. The check uses the radial field, whose flux through any closed surface is $3\times$ the enclosed volume, here $3\cdot(2a)^3 = 24a^3$.

In [ ]:
def flux_through_cube(F, a, n=200):
    """Outward flux of F through the surface of the cube |x|,|y|,|z| <= a."""
    s = -a + (np.arange(n) + 0.5) * (2 * a / n)                 # replace from here with your own version
    U, V = np.meshgrid(s, s, indexing="ij")
    dA = (2 * a / n) ** 2
    total = 0.0
    for axis in range(3):
        for sign in (+1.0, -1.0):
            pts = np.zeros((n * n, 3))
            others = [k for k in range(3) if k != axis]
            pts[:, others[0]] = U.ravel(); pts[:, others[1]] = V.ravel(); pts[:, axis] = sign * a
            total += sign * np.sum(F(pts)[:, axis]) * dA
    return total


for a in (0.5, 1.0):
    val = flux_through_cube(F_radial, a)
    print(f"a = {a}:  flux of (x, y, z) through the cube = {val:.5f}   exact 24 a^3 = {24 * a**3}")
    assert abs(val - 24 * a**3) < 1e-6

## 4. Verify

A field whose flux you can compute by hand through two different closed surfaces: the Coulomb field $\mathbf{F} = \hat{\mathbf{r}}/r^2$. Through a sphere of radius $R$ centred on the origin, $\mathbf{F}\cdot d\mathbf{S} = (1/R^2)\,R^2\sin\theta\,d\theta\,d\phi$ and the flux is $4\pi$, whatever $R$ is. Through the unit cube centred on the origin the integral is harder by hand, but the answer is also $4\pi$. Through any closed surface that does not enclose the origin, it is zero. Chapter 4 turns this fact into Gauss' law. The cube and the displaced sphere are checked numerically here; the displaced sphere needs a fine grid near the side closest to the origin.

In [ ]:
F_coulomb = lambda p: p / np.linalg.norm(p, axis=1, keepdims=True) ** 3
for R in (0.5, 1.0, 2.0):
    c, d = surface_patches(lambda t, p: sphere(t, p, R), (0, np.pi), (0, 2 * np.pi), 200, 400)
    print(f"sphere radius {R}: flux = {flux(F_coulomb, c, d):.5f}", end="   ")
print(f"\nunit cube:            flux = {flux_through_cube(F_coulomb, 1.0, n=400):.5f}   4 pi = {4 * np.pi:.5f}")
shifted = lambda t, p: sphere(t, p, 1.0) + np.array([3.0, 0.0, 0.0])
c, d = surface_patches(shifted, (0, np.pi), (0, 2 * np.pi), 300, 600)
print(f"unit sphere centred at (3, 0, 0), origin outside: flux = {flux(F_coulomb, c, d):.2e}")
assert abs(flux(F_coulomb, c, d)) < 1e-3 and abs(flux_through_cube(F_coulomb, 1.0, n=400) - 4 * np.pi) < 1e-3

## 5. Explore and rebuild

Explore. The work done by $\mathbf{F} = (-y, x)$ around a closed loop came out as $2\pi$ for the unit circle. Compute it for circles of radius $0.5$, $1$, $2$ and for the unit square. Divide each result by the enclosed area. The ratio is the same number every time. Which number, and which of the six reference fields from notebook 00a has that number as its curl? That is Stokes' theorem waiting to be stated in notebook 00d.

Rebuild. Close this notebook. On paper, derive $d\mathbf{S}$ for the sphere from the cross product of the two tangent vectors (worked example 6), and derive the total charge of the ball in worked example 10. In a blank editor, write `vector_line_integral`, `surface_patches` and `flux` from scratch and reproduce the $4\pi$ results.